In [1]:
print("Hello, World!")

Hello, World!


## Evaluation Data

In [84]:
examples = [
    {
        "inputs": {"question": "Is right to privacy a fundamental right in India?"},
        "outputs": {"answer": "Yes. The Supreme Court in K.S. Puttaswamy v. Union of India (2017) held that the right to privacy is a fundamental right protected under Article 21 of the Constitution."}
    },
    {
        "inputs": {"question": "What is anticipatory bail under Section 438 CrPC?"},
        "outputs": {"answer": "Anticipatory bail is a pre-arrest legal protection granted by a court to a person who apprehends arrest for a non-bailable offence."}     
    },
    {
        "inputs": {"question": "What are the essential elements of a valid contract?"},
        "outputs": {"answer": "A valid contract requires offer, acceptance, lawful consideration, competent parties, free consent, and a lawful object."}
    },
    {
        "inputs": {"question": "What is criminal conspiracy under Indian law?"},
        "outputs": {"answer": "Criminal conspiracy is an agreement between two or more persons to commit an illegal act or a legal act by illegal means."}
    },
    {
        "inputs": {"question": "Can Aadhaar be made mandatory for all services?"},
        "outputs": {"answer": "No. Aadhaar can only be mandated where specifically authorized by law, as clarified by the Supreme Court in the Aadhaar judgment."}
    },
    {
        "inputs": {"question": "What does Article 21 of the Constitution protect?"},
        "outputs": {"answer": "Article 21 protects the right to life and personal liberty and has been interpreted broadly to include several related rights."}
    },
    {
        "inputs": {"question": "What is the doctrine of basic structure?"},
        "outputs": {"answer": "The basic structure doctrine prevents Parliament from altering the fundamental framework of the Constitution through amendments."}
    },
    {
        "inputs": {"question": "What is judicial review?"},
        "outputs": {"answer": "Judicial review is the power of courts to examine the constitutionality and legality of legislative and executive actions."}
    },
    {
        "inputs": {"question": "What is the significance of Kesavananda Bharati v. State of Kerala?"},
        "outputs": {"answer": "The case established the basic structure doctrine and limited Parliament's power to amend the Constitution."}
    },
    {
        "inputs": {"question": "What is the punishment for contempt of court?"},
        "outputs": {"answer": "Punishment for contempt of court may include imprisonment, a fine, or both, depending on the nature and severity of the contempt."}
    }
]

In [85]:
from langsmith import Client
from dotenv import load_dotenv

load_dotenv()

client = Client()

dataset_name = "Indian Law Evaluation Datasets"

try:
    dataset = client.read_dataset(dataset_name=dataset_name)
except Exception:
    dataset = client.create_dataset(dataset_name=dataset_name)


client.create_examples(
    dataset_id = dataset.id,
    examples = examples
)


{'example_ids': ['4839f3eb-0526-43bc-8d76-81ac2711a2fd',
  '0654d132-1c60-4748-8928-f8f271aae354',
  'e4319eed-f752-478c-b6aa-ec9fffa873c3',
  '85cb8732-b8d9-4957-bee5-059ae48923bc',
  '3af0f142-b7b5-4d28-9191-c59c3ce51da7',
  '8abce907-14a4-41b0-b90e-9976474392ef',
  '3b9d188f-9aa7-4306-8e5b-8ac07260ce01',
  '1194a702-db1b-4c2b-ae1b-11675da4f630',
  'a1204fcb-6bd7-415e-9b1b-adddc48c8d27',
  '898c6432-95b3-4614-af3f-ea7c6ad23f06'],
 'count': 10,
 'as_of': '2026-06-20T21:40:51.899308965Z'}

## Correctness

In [110]:
from typing_extensions import Annotated, TypedDict

class Correctness(TypedDict):
    explanation: Annotated[str, ... ,"Explanation of the correctness of the answer"]
    correct: Annotated[bool, ... ,"Whether the answer is correct or not"] 
    

correctness_prompt = """
You are an expert legal evaluator.

Your task is to determine whether the AI-generated answer is factually correct compared to the ground truth answer.


Evaluation Criteria:
1. Check whether the generated answer contains the same legal meaning as the ground truth.
2. Minor wording differences are acceptable.
3. The generated answer may contain additional relevant legal information, provided it is not incorrect.
4. Penalize factual inaccuracies, legal misinterpretations, missing key facts, or hallucinated information.
5. Focus on correctness of legal concepts, case references, constitutional provisions, and legal reasoning.

Return your evaluation strictly in the following JSON format:

{
    "correct": <true or false>,
    "explanation": "<brief explanation>"
}

Where:
- correct = true if the answer is substantially correct.
- correct = false if the answer contains incorrect, misleading, or missing critical legal information.
- explanation should briefly justify the decision.
"""


from langchain.chat_models import init_chat_model

llm = init_chat_model(
    model="llama-3.3-70b-versatile",
    model_provider="groq"
)


evaluator_llm = llm.with_structured_output(Correctness)





In [111]:
def correctness_evaluator(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    prompt = f"""
    Ground Truth Answer: {reference_outputs.get('answer', '')}
    
    Generated Answer: {outputs.get('answer', '')}

    input: {inputs.get('question', '')}
    
    """
    grade = evaluator_llm.invoke(
        [
            {"role": "system", "content": correctness_prompt},
            {"role": "user", "content": prompt}
        ]
    )

    return {
        "key": "correctness",
        "score": int(bool(grade["correct"])),
        "comment": grade["explanation"],
    }

In [112]:
from typing_extensions import Annotated, TypedDict

class Relevance(TypedDict):
    explanation: Annotated[str, ... ,"Explanation of the relevance of the answer"]
    relevant: Annotated[bool, ... ,"Whether the answer is relevant or not"] 
    

relevance_prompt = """
You are an expert legal evaluator.

Your task is to assess how relevant the generated answer is to the user's question.

Evaluation Criteria:

1. The answer should directly address the user's question.
2. The answer should remain focused on the legal issue raised.
3. Irrelevant legal concepts, cases, or information should reduce the score.
4. A partially relevant answer should receive a moderate score.
5. A complete and focused answer should receive a high score.


Return your evaluation strictly in the following JSON format:

{
"relevant": <true or false>,
"explanation": "<brief explanation>"
}

Where:
- relevant = true if the answer directly and materially addresses the legal question.
- relevant = false if the answer is off-topic, incomplete in a major way, or mostly irrelevant.

"""



relevance_llm = llm.with_structured_output(Relevance)



In [113]:
def relevance_evaluator(inputs: dict, outputs: dict) -> dict:
    prompt = f"""
    
    
    Generated Answer: {outputs.get('answer', '')}

    input: {inputs.get('question', '')}
    
    """
    grade = relevance_llm.invoke(
        [
            {"role": "system", "content": relevance_prompt},
            {"role": "user", "content": prompt}
        ]
    )

    return {
        "key": "answer_relevance",
        "score": int(bool(grade["relevant"])),
        "comment": grade["explanation"],
    }

## Retriever Relevance

In [114]:
from typing_extensions import Annotated, TypedDict

class Retriever_relevance(TypedDict):
    explanation: Annotated[str, ... ,"Explanation of the relevance of the answer"]
    retrieve_relevance: Annotated[bool, ... ,"Whether the answer is relevant or not"] 
    

retrieve_relevance_prompt = """
You are an expert legal evaluator.

Your task is to determine whether the retrieved context is relevant for answering the user's legal question.

Evaluation Criteria:

1. The retrieved context should contain information that helps answer the question.
2. The context should discuss the same legal issue, statute, principle, or case law as the question.
3. Context containing unrelated legal topics should receive a lower score.
4. Partial relevance should receive a moderate score.
5. Highly relevant context that directly supports answering the question should receive a high score.


Return your evaluation strictly in the following JSON format:

{
"retrieve_relevance": <true or false>,
"explanation": "<brief explanation>"
}

Where:
- retrieve_relevance = true if the retrieved context is useful for answering the legal question.
- retrieve_relevance = false if the retrieved context is unrelated or not materially useful.
"""



Retriever_relevance_llm = llm.with_structured_output(Retriever_relevance)



In [115]:
def retrieval_relevance_evaluator(inputs: dict, outputs: dict) -> dict:
    prompt = f"""
    docs : {outputs.get('retrieved_context', '')}    
    Generated Answer: {outputs.get('answer', '')}

    input: {inputs.get('question', '')}
    
    """
    grade = Retriever_relevance_llm.invoke(
        [
            {"role": "system", "content": retrieve_relevance_prompt},
            {"role": "user", "content": prompt}
        ]
    )

    return {
        "key": "retrieval_relevance",
        "score": int(bool(grade["retrieve_relevance"])),
        "comment": grade["explanation"],
    }


In [116]:
from graph import new_workflow
from langchain.messages import HumanMessage


workflow_input = {
    "messages": [HumanMessage(content="What is Delhi liquor ban supreme court judgement in 2022?")],
    "retriever_docs": [],
    "tools_used": [],
    "draft": "",
    "draft_type": ""
}


result = new_workflow.invoke(workflow_input)


Retriever Tool Called with query: Delhi liquor ban Supreme Court judgment 2022


c:\Users\soodm\Desktop\AI Legal Assistant\.venv\Lib\site-packages\langchain_core\vectorstores\base.py:1054: UserWarning: Method `max_marginal_relevance_search` was called on a vector store equipped with Hybrid capabilities. Since this method cannot make use of Hybrid search, the vector store will fall back to regular vector ANN similarity search.
  docs = self.vectorstore.max_marginal_relevance_search(query, **kwargs_)


In [117]:
input = result['messages']
input = input[0].content if input else ""

In [118]:
print("Input:", input)

Input: What is Delhi liquor ban supreme court judgement in 2022?


In [119]:
generated_answer = result['messages'][-1].content

In [120]:
print(generated_answer)

The Supreme Court of India did not issue a specific judgment regarding a "Delhi liquor ban" in 2022 that is widely recognized. However, there were significant developments related to the Delhi government's liquor policy and its subsequent challenges in the courts.

In 2021, the Delhi government introduced a new excise policy that aimed to privatize the sale of liquor, which faced criticism and legal challenges. The policy was intended to increase revenue and curb illegal liquor trade. However, it was met with opposition, leading to a ban on certain aspects of the policy.

In 2022, the Delhi High Court upheld the ban on the sale of liquor in certain areas, citing public health and safety concerns. The Supreme Court's involvement in this matter primarily revolved around appeals against the High Court's decisions.

If you are looking for a specific judgment or more detailed information about a particular case or aspect of the liquor policy, please provide additional details, and I can ass

In [121]:
def target(inputs: dict) -> dict:
    workflow_input = {
        "messages": [HumanMessage(content=inputs["question"])],
        "retriever_docs": [],
        "tools_used": [],
        "draft": "",
        "draft_type": ""
    }
    result = new_workflow.invoke(workflow_input)
    answer = result['messages'][-1].content if result.get('messages') else "No response generated"
    retrieved_context = "\n\n".join(
        doc.page_content for doc in result.get('retriever_docs', []) if hasattr(doc, 'page_content')
    )
    return {"answer": answer, "retrieved_context": retrieved_context}

In [122]:
target({"question": "Can you tell me about the Land Acquisition Act supreme court judgment in 2022?"})

Retriever Tool Called with query: Land Acquisition Act Supreme Court judgment 2022


c:\Users\soodm\Desktop\AI Legal Assistant\.venv\Lib\site-packages\langchain_core\vectorstores\base.py:1054: UserWarning: Method `max_marginal_relevance_search` was called on a vector store equipped with Hybrid capabilities. Since this method cannot make use of Hybrid search, the vector store will fall back to regular vector ANN similarity search.
  docs = self.vectorstore.max_marginal_relevance_search(query, **kwargs_)


{'answer': "In 2022, the Supreme Court of India delivered significant judgments concerning the Land Acquisition Act, particularly focusing on the interpretation of provisions related to the lapse of acquisition proceedings and the necessity of adhering to statutory requirements.\n\n### Key Highlights from the Judgments:\n\n1. **Lapse of Acquisition Proceedings**:\n   - The Court emphasized that if land acquisition proceedings initiated under the Land Acquisition Act do not comply with specific statutory requirements, they may be deemed to have lapsed. This includes situations where no award has been made within five years of the initiation of the proceedings, or where physical possession has not been taken, and compensation has not been paid.\n\n2. **Public Purpose Requirement**:\n   - The judgments reiterated that land acquisition must be for a public purpose, which is a fundamental requirement under the Act. The Court scrutinized cases where land was acquired for private entities and

In [123]:
experiments = client.evaluate(
    target,
    data=dataset_name,
    evaluators=[
        correctness_evaluator,
        relevance_evaluator
    ],
    experiment_prefix="Indian Law Evaluation by groq/llama-3.3-70b-versatile",
    metadata={
        "dataset_id": dataset.id
    }
)

experiments.to_pandas()

View the evaluation results for experiment: 'Indian Law Evaluation by groq/llama-3.3-70b-versatile-7a3570db' at:
https://smith.langchain.com/o/5e69aaac-f62f-4b76-a26a-ed4449032676/datasets/e48b8ef0-aa16-4b4e-a00c-3d2965a73869/compare?selectedSessions=3db76085-81b8-43e7-bc64-55ab0ef0b375




2it [00:10,  5.09s/it]

Retriever Tool Called with query: Kesavananda Bharati v. State of Kerala significance


c:\Users\soodm\Desktop\AI Legal Assistant\.venv\Lib\site-packages\langchain_core\vectorstores\base.py:1054: UserWarning: Method `max_marginal_relevance_search` was called on a vector store equipped with Hybrid capabilities. Since this method cannot make use of Hybrid search, the vector store will fall back to regular vector ANN similarity search.
  docs = self.vectorstore.max_marginal_relevance_search(query, **kwargs_)
12it [01:05,  6.54s/it]

Retriever Tool Called with query: Kesavananda Bharati v. State of Kerala significance


c:\Users\soodm\Desktop\AI Legal Assistant\.venv\Lib\site-packages\langchain_core\vectorstores\base.py:1054: UserWarning: Method `max_marginal_relevance_search` was called on a vector store equipped with Hybrid capabilities. Since this method cannot make use of Hybrid search, the vector store will fall back to regular vector ANN similarity search.
  docs = self.vectorstore.max_marginal_relevance_search(query, **kwargs_)
20it [02:26,  7.34s/it]


,inputs.question,outputs.answer,outputs.retrieved_context,error,reference.answer,feedback.correctness,feedback.answer_relevance,execution_time,example_id,id
0,What are the essential elements of a valid con...,The essential elements of a valid contract und...,,None,"A valid contract requires offer, acceptance, l...",1,1,4.419258,0654d132-1c60-4748-8928-f8f271aae354,019ee714-45ef-7972-87d5-e563db537830
1,Can Aadhaar be made mandatory for all services?,The issue of making Aadhaar mandatory for vari...,,None,No. Aadhaar can only be mandated where specifi...,1,1,4.080207,1194a702-db1b-4c2b-ae1b-11675da4f630,019ee714-599f-7740-b270-b0abf78af555
2,What is the significance of Kesavananda Bharat...,The case of **Kesavananda Bharati v. State of ...,,None,The case established the basic structure doctr...,1,1,7.183203,3af0f142-b7b5-4d28-9191-c59c3ce51da7,019ee714-6da9-7c01-b71f-a7e184f460b7
3,What is criminal conspiracy under Indian law?,Criminal conspiracy under Indian law is define...,,None,Criminal conspiracy is an agreement between tw...,1,1,4.226944,3b9d188f-9aa7-4306-8e5b-8ac07260ce01,019ee714-8c6d-7d00-8166-ec878dae2230
4,Is right to privacy a fundamental right in India?,"Yes, the right to privacy is recognized as a f...",,None,Yes. The Supreme Court in K.S. Puttaswamy v. U...,1,1,2.429436,4839f3eb-0526-43bc-8d76-81ac2711a2fd,019ee714-a01b-76e0-910c-0d5c6791903d
5,What is judicial review?,Judicial review is the power of the judiciary ...,,None,Judicial review is the power of courts to exam...,1,1,3.872441,85cb8732-b8d9-4957-bee5-059ae48923bc,019ee714-ad6c-77e3-975c-6e3ae69a4e34
6,What is the punishment for contempt of court?,Contempt of court in India is governed by the ...,,None,Punishment for contempt of court may include i...,1,1,3.796054,898c6432-95b3-4614-af3f-ea7c6ad23f06,019ee714-bf81-7a42-81f8-231d2856774d
7,What is anticipatory bail under Section 438 CrPC?,Anticipatory bail is a provision under Section...,,None,Anticipatory bail is a pre-arrest legal protec...,1,1,4.004345,8abce907-14a4-41b0-b90e-9976474392ef,019ee714-d12a-7763-9a42-416987445561
8,What does Article 21 of the Constitution protect?,Article 21 of the Constitution of India provid...,,None,Article 21 protects the right to life and pers...,1,1,3.315413,a1204fcb-6bd7-415e-9b1b-adddc48c8d27,019ee714-e452-77c0-8f52-418f59e3fc3b
9,What is the doctrine of basic structure?,The doctrine of basic structure is a constitut...,,None,The basic structure doctrine prevents Parliame...,1,1,4.103178,e4319eed-f752-478c-b6aa-ec9fffa873c3,019ee714-f473-7e53-a005-84a79bc137ad


In [124]:
dataset = client.read_dataset(dataset_name=dataset_name)

for example in client.list_examples(dataset_id=dataset.id):
    print(example.inputs)
    print(example.outputs)
    break

{'question': 'What are the essential elements of a valid contract?'}
{'answer': 'A valid contract requires offer, acceptance, lawful consideration, competent parties, free consent, and a lawful object.'}
